In [1]:
import pandas as pd
import numpy as np
import hashlib

# 1. Generate 10,000 initial digital economy transaction records
np.random.seed(42)
data = {
    'Transaction_ID': [f"TXN_{i:06d}" for i in range(1, 10001)], # Unique transaction ID
    'User_Age': np.random.randint(18, 65, 10000),
    'Payment_Amount': np.round(np.random.uniform(10.0, 1000.0, 10000), 2) # Normal amounts only have 2 decimal places
}
df_original = pd.DataFrame(data)

print(f"[Original Data Generated] Total {len(df_original)} rows.")
display(df_original.head(3))

# The absolute secret of the gateway (salt value that must never be leaked)
SYSTEM_SECRET_KEY = "SOTON_COMP_GATEWAY_2026"

def inject_robust_watermark(df, buyer_id):
    """Randomly sprinkle the buyer's invisible features into the data ocean."""
    df_wm = df.copy()
    watermark_count = 0
    
    for index, row in df_wm.iterrows():
        txn_id = row['Transaction_ID']
        
        # 1. Core cryptography: Combine Transaction ID + System Secret + Buyer ID to calculate SHA-256 hash
        hash_str = f"{txn_id}_{SYSTEM_SECRET_KEY}_{buyer_id}"
        hash_hex = hashlib.sha256(hash_str.encode()).hexdigest()
        
        # 2. Convert the hash value to a number and take modulo 100.
        # This means we only inject the watermark into about 5% of the data rows, extremely hidden!
        if int(hash_hex, 16) % 100 < 5: 
            # 3. Inject a tiny perturbation: add 0.0088 (no impact on macro statistics)
            original_amount = row['Payment_Amount']
            df_wm.at[index, 'Payment_Amount'] = round(original_amount + 0.0088, 4)
            watermark_count += 1
            
    print(f"[Watermark Injected] Customized for {buyer_id}.")
    print(f"[Stealth Report] Out of 10,000 rows, only {watermark_count} rows were quietly modified.")
    return df_wm

# Simulate the gateway selling data to Alice
df_alice = inject_robust_watermark(df_original, "Alice_Corp")

# Simulate Alice leaking the data, with "destructive cleaning" before the leak
print("[Hacker Action] Alice is attempting to destroy the watermark...")

# 1. Randomly delete 5,000 rows of data (severe destruction)
df_leaked = df_alice.sample(frac=0.5, random_state=99) 

# 2. Completely shuffle the original table order
df_leaked = df_leaked.sample(frac=1).reset_index(drop=True)

print(f"[Destruction Complete] The leaked data now only has {len(df_leaked)} rows and is totally scrambled.")

def verify_suspect(leaked_df, suspect_id):
    """Examine the suspect: calculate expected watermarks vs. actually found watermarks."""
    expected_marks = 0
    found_marks = 0
    
    for index, row in leaked_df.iterrows():
        txn_id = row['Transaction_ID']
        amount = row['Payment_Amount']
        
        # Reconstruct the suspect's hash feature
        hash_str = f"{txn_id}_{SYSTEM_SECRET_KEY}_{suspect_id}"
        if int(hashlib.sha256(hash_str.encode()).hexdigest(), 16) % 100 < 5:
            expected_marks += 1 # Theoretically, this row should have a watermark
            
            # Check if the fractional part of the amount ends in 88
            fractional_part = round((amount * 10000) % 100)
            if fractional_part == 88:
                found_marks += 1
                
    # Calculate match rate (found marks / expected marks)
    if expected_marks == 0: return 0
    confidence = (found_marks / expected_marks) * 100
    return confidence

# ----------------- Begin Trial -----------------
print("[System] Starting the digital blind watermark tracing engine...")
suspects = ["Bob_Analytics", "Charlie_Data", "Alice_Corp"]

for suspect in suspects:
    match_rate = verify_suspect(df_leaked, suspect)
    print(f"[Scanning] Checking suspect [{suspect}]... Genetic match rate: {match_rate:.2f}%")
    
    if match_rate > 80.0: # As long as feature retention is over 80%, it's solid proof
        print(f"    >>> [Solid Proof] Leak source locked: {suspect}! Ready for legal action.\n")

[Original Data Generated] Total 10000 rows.


,Transaction_ID,User_Age,Payment_Amount
0,TXN_000001,56,393.75
1,TXN_000002,46,329.96
2,TXN_000003,32,935.81


[Watermark Injected] Customized for Alice_Corp.
[Stealth Report] Out of 10,000 rows, only 489 rows were quietly modified.
[Hacker Action] Alice is attempting to destroy the watermark...
[Destruction Complete] The leaked data now only has 5000 rows and is totally scrambled.
[System] Starting the digital blind watermark tracing engine...
[Scanning] Checking suspect [Bob_Analytics]... Genetic match rate: 6.87%
[Scanning] Checking suspect [Charlie_Data]... Genetic match rate: 6.64%
[Scanning] Checking suspect [Alice_Corp]... Genetic match rate: 100.00%
    >>> [Solid Proof] Leak source locked: Alice_Corp! Ready for legal action.



In [ ]:
#网关不能等几万条数据全准备好了再处理。你需要用流式 JSON 解析器（比如 Python 的 ijson，Go 的 json.Decoder），读到一行数据，就处理一行，发走一行。

#绝对不能把 SYSTEM_SECRET_KEY 写死在代码里。网关应该在启动时从 KMS（密钥管理服务）、HashiCorp Vault 或环境变量中读取这个盐值，并缓存在内存中。

#网关在处理请求头（Headers）时，从 Authorization Token 中提取出当前调用者的 buyer_id，传给水印注入模块

#你可以对 txn_id 进行截断再哈希（比如只取前 8 位），或者用更轻量级的非加密哈希算法（如 MurmurHash3），
#只要能保证雪崩效应和均匀分布，并结合你的 SECRET_KEY（比如做 HMAC）即可。
#

In [ ]:
import hashlib
import json
from fastapi import Request, Response
from starlette.middleware.base import BaseHTTPMiddleware

class WatermarkMiddleware(BaseHTTPMiddleware):
    def __init__(self, app, secret_key: str):
        super().__init__(app)
        self.secret_key = secret_key

    async def dispatch(self, request: Request, call_next):
        # 1. Extract buyer identity (assuming it is obtained from the request header)
        buyer_id = request.headers.get("X-Buyer-ID", "Unknown_Buyer")

        # 2. Call the backend service to get the original response stream
        response = await call_next(request)

        # 3. Intercept the response body for streaming modification 
        # (This is simplified logic assuming we are processing a JSON array)
        # In actual production, it is recommended to use StreamingResponse with an iterator generator
        if response.headers.get("content-type") == "application/json":
            body = b""
            async for chunk in response.body_iterator:
                body += chunk
            
            # Parse the original data
            data = json.loads(body)
            
            # Inject the watermark
            for item in data:
                txn_id = item.get("Transaction_ID")
                if txn_id:
                    # Calculate the cryptographic feature
                    hash_str = f"{txn_id}_{self.secret_key}_{buyer_id}"
                    hash_hex = hashlib.sha256(hash_str.encode()).hexdigest()
                    
                    # Trigger injection with a 5% probability
                    if int(hash_hex, 16) % 100 < 5:
                        original_amount = item.get("Payment_Amount", 0)
                        # Inject a tiny perturbation
                        item["Payment_Amount"] = round(original_amount + 0.0088, 4)
            
            # Repackage into a new response
            new_body = json.dumps(data).encode("utf-8")
            return Response(content=new_body, media_type="application/json")
        
        return response